# Cálculo dos custos de conexão dos latifúndios de MG

## Imports

In [2]:
import os
import pandas as pd
import geopandas as gpd
import gcsfs
from dotenv import find_dotenv, load_dotenv


## Paths

In [6]:
dotenv_path = find_dotenv()
load_dotenv(dotenv_path)

project_root = os.path.dirname(dotenv_path)

KEY_PATH_RELATIVE = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
if KEY_PATH_RELATIVE:
    KEY_PATH_ABSOLUTE = os.path.join(project_root, KEY_PATH_RELATIVE)
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = KEY_PATH_ABSOLUTE
    print(f"Credenciais carregadas com sucesso de: {KEY_PATH_ABSOLUTE}")
else:
    print(
        "Aviso: A variável GOOGLE_APPLICATION_CREDENTIALS não foi encontrada no arquivo .env"
    )
    print("O acesso ao GCS pode falhar.")

BUCKET_NAME = os.getenv("GCS_BUCKET_NAME")
DIR_RAW_GRID = "../data/raw/electrical_grid/"
CANDIDATOS_GCS_PATH = f"gs://{BUCKET_NAME}/processed/candidatos_solares_mg_limpo.gpkg"
TEMP_FILE = "temp_candidatos_mg.gpkg"

Credenciais carregadas com sucesso de: g:\BackupC\Faculdade\UFRJ\TCC\analise-geografica-energia-solar-brasil\.secrets/tcc-usinas-solares-brasil-29b2750a20f8.json


## Carregamento de Dados

In [ ]:
fs = gcsfs.GCSFileSystem()
fs.get(CANDIDATOS_GCS_PATH, TEMP_FILE)
candidatos_gdf = gpd.read_file(TEMP_FILE, engine="fiona")
os.remove(TEMP_FILE)
print(f"Candidatos carregados: {len(candidatos_gdf):,} propriedades.")

subestacoes_gdf = gpd.read_file(os.path.join(DIR_RAW_GRID, "Subestações_-_Base_Existente.shp"))
linhas_gdf = gpd.read_file(os.path.join(DIR_RAW_GRID, "Linhas_de_Transmissão_-_Base_Existente.shp"))
print("Dados carregados com sucesso!")

Candidatos carregados: 8,150 propriedades.
Dados carregados com sucesso!


## Processamento dos dados

### Remoção de linhas de baixa tensão

In [10]:
linhas_gdf["Tensao_num"] = pd.to_numeric(linhas_gdf["Tensao"], errors="coerce")
linhas_alta_tensao = linhas_gdf[linhas_gdf["Tensao"] >= 230].copy()

### Reprojeção dos dados para cálculo de distâncias

In [11]:
CRS_METRICO = "EPSG:31983"
candidatos_proj = candidatos_gdf.to_crs(CRS_METRICO)
subestacoes_proj = subestacoes_gdf.to_crs(CRS_METRICO)
linhas_proj = linhas_alta_tensao.to_crs(CRS_METRICO)

## Cálculos

### Distância da rede elétrica

In [13]:
# Juntando geometrias de subestações (Pontos) e linhas (Linhas) num só GeoDataFrame
infra_geom = pd.concat([
    subestacoes_proj[['geometry']], 
    linhas_proj[['geometry']]
], ignore_index=True)

infra_gdf = gpd.GeoDataFrame(infra_geom, geometry='geometry', crs=CRS_METRICO)

In [15]:
# Extraindo centróides dos latifúndios para cálculo de distâncias
candidatos_centroides = candidatos_proj.copy()
candidatos_centroides['geometry'] = candidatos_centroides.geometry.centroid

In [16]:
# O sjoin_nearest encontra a geometria da infraestrutura mais próxima e retorna a distância
candidatos_com_distancia = gpd.sjoin_nearest(
    candidatos_centroides, 
    infra_gdf, 
    how='left', 
    distance_col='distancia_metros'
)

In [17]:
# Remoção de possíveis duplicatas (duas linhas com esma distância do terreno)
candidatos_com_distancia = candidatos_com_distancia[~candidatos_com_distancia.index.duplicated(keep='first')]

### Custo de conexão

In [18]:
# Conversão de metros pra km
candidatos_com_distancia['distancia_km'] = candidatos_com_distancia['distancia_metros'] / 1000

In [19]:
# PREMISSA DE CUSTO: R$ 1.500.000 por Quilômetro de Linha de Transmissão construída (Valor estimado base EPE)
CUSTO_POR_KM = 1500000 
candidatos_com_distancia['custo_conexao_rs'] = candidatos_com_distancia['distancia_km'] * CUSTO_POR_KM

In [20]:
candidatos_proj['distancia_km'] = candidatos_com_distancia['distancia_km']
candidatos_proj['custo_conexao_rs'] = candidatos_com_distancia['custo_conexao_rs']

In [21]:
print("-" * 50)
print("TOP 5 LATIFÚNDIOS MAIS BEM LOCALIZADOS (Menor Custo)")
print("-" * 50)
print(candidatos_proj[['id_imovel', 'nome_imovel', 'area_util_ha', 'distancia_km', 'custo_conexao_rs']].sort_values(by='distancia_km').head())

--------------------------------------------------
TOP 5 LATIFÚNDIOS MAIS BEM LOCALIZADOS (Menor Custo)
--------------------------------------------------
          id_imovel                             nome_imovel  area_util_ha  \
6260  4040630384580                 Fazenda Capão - Parte 1    189.086017   
3292  4210490097847  Fazenda São José do Fecho - PARCELA 01    378.172035   
3897  4140180093266         Fazenda Santa Maria - Parcela 1    189.086017   
1036  4140850160474               FAZENDA FORMATO - Parte 1    378.172035   
8027  4230760105708     FAZENDA SENHOR DO BOM FIM - Parte 1   3119.919288   

      distancia_km  custo_conexao_rs  
6260      0.003725       5588.182867  
3292      0.006303       9454.304953  
3897      0.007118      10676.705936  
1036      0.008143      12214.850270  
8027      0.008213      12320.027232  


## Geração do input do solver (matriz)

In [24]:
# Preparando DataFrame final para solver de otimização
df_solver = pd.DataFrame(candidatos_proj.drop(columns='geometry'))
colunas_solver = [
    'id_imovel', 
    'nome_imovel', 
    'area_util_ha', 
    'potencial_mw', 
    'distancia_km', 
    'custo_conexao_rs'
]
df_solver_final = df_solver[colunas_solver].copy()
df_solver_final = df_solver_final.sort_values(by='custo_conexao_rs').reset_index(drop=True)

In [26]:
df_solver_final.head(5)

,id_imovel,nome_imovel,area_util_ha,potencial_mw,distancia_km,custo_conexao_rs
0,4040630384580,Fazenda Capão - Parte 1,189.086017,52.523894,0.003725,5588.182867
1,4210490097847,Fazenda São José do Fecho - PARCELA 01,378.172035,105.047787,0.006303,9454.304953
2,4140180093266,Fazenda Santa Maria - Parcela 1,189.086017,52.523894,0.007118,10676.705936
3,4140850160474,FAZENDA FORMATO - Parte 1,378.172035,105.047787,0.008143,12214.850270
4,4230760105708,FAZENDA SENHOR DO BOM FIM - Parte 1,3119.919288,866.644247,0.008213,12320.027232


In [25]:
# Salvando resultado final em CSV e enviando para GCS
TEMP_CSV = "temp_solver_input_mg.csv"
df_solver_final.to_csv(TEMP_CSV, index=False, sep=';', decimal=',')

SOLVER_OUTPUT_PATH = f"gs://{BUCKET_NAME}/solver_inputs/candidatos_solver_mg.csv"
fs.put(TEMP_CSV, SOLVER_OUTPUT_PATH)
os.remove(TEMP_CSV)

print(f"Arquivo salvo com sucesso na nuvem: {SOLVER_OUTPUT_PATH}")

Arquivo salvo com sucesso na nuvem: gs://tcc-usinas-solares-brasil-dados/solver_inputs/candidatos_solver_mg.csv
